[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C05_Safety_Evals_Course/01_risk_frameworks/01_risk_frameworks.ipynb)

# 模块 01 · 能力阈值追踪器（Capability Threshold Tracker）

**算力：CPU only** —— 纯 numpy / pandas / matplotlib，无需 GPU、无需下载模型、无需任何 API key。

配套讲解：`01_讲解.html`。本 notebook 把安全框架（Anthropic RSP / OpenAI Preparedness / Google DeepMind FSF）的核心机制——
**能力阈值（capability threshold）→ 预警 buffer → if-then 承诺**——落成可运行的代码 [Shevlane 2023]：

1. 合成 5 代模型 × 4 个风险域的评测分数时间序列（含不确定度）；
2. 实现三色判定 `threshold_status`（green / yellow / red）；
3. 画带置信区间误差棒的能力增长轨迹 + 阈值线；
4. **外推预警**：线性/指数拟合外推下一代分数与首次穿越时间；
5. **scorecard 生成器**：Preparedness 风格的 markdown 汇总；
6. 敏感性分析：buffer 大小 vs 预警提前量。

> ⚠️ **数据声明**：所有"评测分数"均为**手工构造的合成数据**，仅用于演示治理机制的测量逻辑，
> 不对应任何真实模型；四个风险域只作为治理标签出现，不涉及任何域内有害知识。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)
Z = 1.96  # 95% 置信系数

# 四大重点风险域（治理标签）与各域的能力阈值 T、预警 buffer B
DOMAINS    = ["CBRN", "Cyber", "Persuasion", "Autonomy"]
THRESHOLDS = {"CBRN": 0.60, "Cyber": 0.70, "Persuasion": 0.65, "Autonomy": 0.75}
BUFFERS    = {"CBRN": 0.10, "Cyber": 0.10, "Persuasion": 0.10, "Autonomy": 0.12}

# 合成数据：5 代模型（G1..G5）在每个域上的评测分数（0~1）与标准误 se
# 故意构造不同形态：CBRN 缓慢增长 / Cyber 快速逼近并越线 / Persuasion 进入预警区 / Autonomy 在 G5 出现能力跳变
GENS = np.array([1, 2, 3, 4, 5])
SCORES = {
    "CBRN":       np.array([0.10, 0.15, 0.20, 0.26, 0.31]),
    "Cyber":      np.array([0.25, 0.34, 0.45, 0.58, 0.71]),
    "Persuasion": np.array([0.20, 0.28, 0.37, 0.46, 0.54]),
    "Autonomy":   np.array([0.08, 0.12, 0.18, 0.27, 0.52]),  # 注意 G4→G5 的跳变
}
SES = {"CBRN": 0.03, "Cyber": 0.03, "Persuasion": 0.03, "Autonomy": 0.05}

df = pd.DataFrame(
    [{"gen": int(g), "domain": d, "score": SCORES[d][i], "se": SES[d]}
     for d in DOMAINS for i, g in enumerate(GENS)]
)
df.pivot(index="gen", columns="domain", values="score")

## 1 · 三色判定规则（threshold status）

把讲解第 4 节的判定规则写成代码。给定分数点估计 $\hat c$、其 95% 置信半宽 $ci = z_\alpha \hat\sigma$、阈值 $T$、buffer $B$：

- 🔴 **red**：$\hat c \ge T$ —— 点估计已越过阈值，触发框架承诺（if-then 的 "if" 成立）；
- 🟡 **yellow**：$\hat c + ci \ge T - B$ —— **置信上界**进入预警区 $[T-B,\,T)$，启动缓解准备；
- 🟢 **green**：其余。

用置信上界判黄灯是治理上的保守原则：对"可能已接近危险"的不确定性按接近危险处理。
下面先用一个内联实现跑通主流程（**练习 1** 会要求你自己重新实现它）。

In [ ]:
def status_of(score, se, threshold, buffer, z=Z):
    '''主流程用的内联三色判定（练习 1 要求你重写一个等价函数）'''
    if score >= threshold:
        return "red"
    if score + z * se >= threshold - buffer:
        return "yellow"
    return "green"

# 最新一代（G5）各域状态
latest = df[df.gen == 5].copy()
latest["threshold"] = latest.domain.map(THRESHOLDS)
latest["buffer"]    = latest.domain.map(BUFFERS)
latest["status"]    = [status_of(r.score, r.se, r.threshold, r.buffer) for r in latest.itertuples()]
print(latest[["domain", "score", "se", "threshold", "buffer", "status"]].to_string(index=False))

# 能力增长轨迹：误差棒(95% CI) + 阈值线(红) + 预警线(橙)
COLORS = {"green": "#2e9e4f", "yellow": "#d99a00", "red": "#cc3344"}
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
for ax, d in zip(axes.ravel(), DOMAINS):
    s, T, B = SCORES[d], THRESHOLDS[d], BUFFERS[d]
    st = status_of(s[-1], SES[d], T, B)
    ax.errorbar(GENS, s, yerr=Z * SES[d], fmt="o-", capsize=4, color="#3366cc", label="score (95% CI)")
    ax.axhline(T, color="#cc3344", ls="-", lw=1.5, label=f"threshold T={T}")
    ax.axhline(T - B, color="#d99a00", ls="--", lw=1.5, label=f"alert line T-B={T-B:.2f}")
    ax.set_title(f"{d}  (G5 status: {st})", color=COLORS[st])
    ax.set_ylim(0, 1); ax.set_xlabel("model generation"); ax.set_ylabel("eval score")
    ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Capability trajectories vs thresholds (synthetic data)")
fig.tight_layout(); plt.show()

## 2 · 外推预警（forecast-based early warning）

框架要求评测**在越线之前**报警（能力跳变 × 评测滞后，见讲解 §4）。最朴素的前瞻工具是趋势外推：
对每个域分别做**线性拟合** $\hat c(t)=at+b$ 与**指数拟合** $\hat c(t)=c\,e^{kt}$（对 $\log \hat c$ 做线性回归），
外推下一代（G6）分数，并解出线性模型下的**首次穿越代数** $t^\* = (T-b)/a$。

> ⚠️ **外推的诚实声明**：外推假设"历史增长模式延续"，而这恰恰是前沿能力最不可靠的假设——
> Autonomy 域在 G4→G5 的跳变就会让基于 G1–G4 的任何拟合严重低估 G5。
> 因此外推结果只能作为**预警的下界参考**（"最晚也要在 $t^\*$ 前准备好缓解"），绝不能当作"在 $t^\*$ 之前都安全"的证据。
> 线性与指数预测的差距本身就是模型不确定性的信号。

In [ ]:
rows = []
for d in DOMAINS:
    s, T = SCORES[d], THRESHOLDS[d]
    a, b = np.polyfit(GENS, s, 1)            # 线性: score = a*gen + b
    k, logc = np.polyfit(GENS, np.log(s), 1)  # 指数: log(score) = k*gen + log(c)
    lin_g6 = a * 6 + b
    exp_g6 = float(np.exp(logc + k * 6))
    eta = (T - b) / a if a > 0 else np.inf    # 线性模型下首次穿越代数
    rows.append({"domain": d, "slope_a": round(a, 3),
                 "linear_G6": round(lin_g6, 3), "exp_G6": round(min(exp_g6, 1.0), 3),
                 "threshold": T, "crossing_gen(linear)": round(eta, 2)})
forecast = pd.DataFrame(rows)
print(forecast.to_string(index=False))
print()
for _, r in forecast.iterrows():
    gap = abs(r["exp_G6"] - r["linear_G6"])
    flag = "  <-- 线性/指数分歧大，外推不可靠" if gap > 0.08 else ""
    print(f"{r['domain']:11s} 预计首次穿越 ~G{r['crossing_gen(linear)']:.1f}"
          f"（线性外推；指数增长下会更早）{flag}")

## 3 · Scorecard 生成器与 buffer 敏感性分析

**Scorecard**：OpenAI Preparedness 风格——把各域的当前状态汇总成一张对外/对治理机构可读的表
（真实 scorecard 还区分缓解前/缓解后等级，这里只做单列演示；**练习 3** 实现可复用的 `scorecard_md`）。

**敏感性分析**：讲解 §4 推出线性增长下的预警提前量
$\Delta t = \dfrac{B + z_\alpha\hat\sigma}{a}$ —— buffer 越大预警越早，但增长越快（$a$ 大）同样的 buffer 换到的时间越短。
我们画出每个域"buffer 大小 → 提前量（代数）"的曲线：它就是**误报成本与漏报灾难之间的权衡曲线**。

In [ ]:
SYM = {"green": "🟢", "yellow": "🟡", "red": "🔴"}

# --- Preparedness 风格 scorecard（markdown 文本）---
lines = ["# Frontier Capability Scorecard — Gen 5 (synthetic)", "",
         "| 风险域 | 分数 (95% CI) | 阈值 T | 预警线 T-B | 状态 |", "|---|---|---|---|---|"]
for r in latest.itertuples():
    ci = Z * r.se
    lines.append(f"| {r.domain} | {r.score:.2f} ± {ci:.2f} | {r.threshold:.2f} "
                 f"| {r.threshold - r.buffer:.2f} | {SYM[r.status]} {r.status} |")
lines += ["", "> red = 点估计越过阈值，触发框架承诺；yellow = 置信上界进入预警区；数据为合成演示。"]
scorecard_text = "\n".join(lines)
print(scorecard_text)

# --- buffer 敏感性：提前量 Δt = (B + z*se) / a ---
b_grid = np.linspace(0.0, 0.25, 60)
plt.figure(figsize=(7, 4.5))
for d in DOMAINS:
    a, _ = np.polyfit(GENS, SCORES[d], 1)
    lead = (b_grid + Z * SES[d]) / a
    plt.plot(b_grid, lead, label=f"{d} (slope a={a:.3f})")
plt.axvline(0.10, color="gray", ls=":", lw=1, label="B=0.10 (default)")
plt.xlabel("buffer size B"); plt.ylabel("warning lead time Δt (generations)")
plt.title("Buffer size vs early-warning lead time (linear-growth model)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()
print("解读：增长最快的域（Cyber）同样 buffer 换到的预警时间最短；"
      "buffer 加大虽延长预警，但会增加误报成本（讲解 §4 的不对称权衡）。")

## ✏️ 练习 1：实现 `threshold_status`

实现框架的三色判定（与主流程 `status_of` 等价，但接口不同：直接传**置信半宽** `ci` 而不是 `se`）：

- `score >= threshold` → 返回 `"red"`；
- 否则 `score + ci >= threshold - buffer` → 返回 `"yellow"`；
- 否则返回 `"green"`。

提示：注意判定用 `>=`（**恰好压线按更危险的一档处理**——保守原则），两个 if 加一个 return 即可（< 10 行）。

In [ ]:
def threshold_status(score, ci, threshold, buffer):
    '''三色判定：score 点估计, ci 置信半宽, threshold 阈值 T, buffer 预警宽度 B
    返回 "green" / "yellow" / "red"'''
    # TODO: red —— 点估计越过(或恰好达到)阈值
    # TODO: yellow —— 置信上界 score+ci 进入(或恰好达到)预警区 [T-B, T)
    # TODO: green —— 其余
    raise NotImplementedError

In [ ]:
# 自测：全过则打印 ✅
assert threshold_status(0.75, 0.05, 0.70, 0.10) == "red"      # 明显越线
assert threshold_status(0.70, 0.00, 0.70, 0.10) == "red"      # 点估计恰好压阈值线 -> red
assert threshold_status(0.55, 0.05, 0.70, 0.10) == "yellow"   # 上界 0.60 恰好压预警线 -> yellow
assert threshold_status(0.60, 0.00, 0.70, 0.10) == "yellow"   # 点估计恰好在预警线上
assert threshold_status(0.50, 0.05, 0.70, 0.10) == "green"    # 上界 0.55 < 0.60
assert threshold_status(0.05, 0.02, 0.70, 0.10) == "green"    # 远离阈值
# 与主流程内联实现在 G5 数据上逐域一致
for r in latest.itertuples():
    assert threshold_status(r.score, Z * r.se, r.threshold, r.buffer) == r.status
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `crossing_eta` —— 线性外推首次穿越

实现线性拟合外推穿越时间的代数：对 `(gens, scores)` 做最小二乘直线拟合 $\hat c(t) = a t + b$，
返回首次穿越阈值的代数 $t^\* = (T - b)/a$；若拟合斜率 $a \le 0$（能力不增长，永不穿越）返回 `None`。

提示：`a, b = np.polyfit(gens, scores, 1)`；总共 ~5 行。在**严格线性**的构造序列上结果应精确到浮点误差。

In [ ]:
def crossing_eta(scores, gens, threshold):
    '''线性拟合 score ≈ a*gen + b，返回首次穿越 threshold 的代数 (threshold-b)/a；
    斜率 a <= 0 时返回 None'''
    # TODO: 1) np.polyfit 拟合得到 a, b
    # TODO: 2) a <= 0 -> return None
    # TODO: 3) return (threshold - b) / a
    raise NotImplementedError

In [ ]:
# 自测：构造严格线性序列 score = 0.1*gen + 0.05
gens_t   = [1, 2, 3, 4, 5]
scores_t = [0.15, 0.25, 0.35, 0.45, 0.55]
assert abs(crossing_eta(scores_t, gens_t, 0.85) - 8.0) < 1e-6   # 未来穿越：(0.85-0.05)/0.1 = 8
assert abs(crossing_eta(scores_t, gens_t, 0.35) - 3.0) < 1e-6   # 历史上已在 G3 恰好压线
assert crossing_eta([0.5, 0.4, 0.3, 0.2, 0.1], gens_t, 0.8) is None  # 能力下降 -> 永不穿越
eta_cyber = crossing_eta(list(SCORES["Cyber"]), list(GENS), THRESHOLDS["Cyber"])
assert eta_cyber is not None and 4.0 < eta_cyber < 6.0          # 与第 2 节的预测一致
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `scorecard_md` —— scorecard 生成器

把 `{域名: 状态}` 字典汇总成 Preparedness 风格的 markdown 表字符串，形如：

```
| 风险域 | 状态 |
|---|---|
| CBRN | 🟢 green |
| Cyber | 🔴 red |
```

要求：①含表头行 `| 风险域 | 状态 |` 与分隔行；②每个域一行，状态用 `SYM` 映射的符号 + 文字；③行间用 `\n` 连接。提示：列表推导 + `"\n".join(...)`，~6 行。

In [ ]:
SYM = {"green": "🟢", "yellow": "🟡", "red": "🔴"}

def scorecard_md(domain_statuses):
    '''domain_statuses: dict 域名 -> "green"/"yellow"/"red"，返回 markdown 表字符串'''
    # TODO: 表头两行 + 每个域一行 "| 域名 | 符号 状态 |"
    raise NotImplementedError

In [ ]:
# 自测
statuses = {"CBRN": "green", "Cyber": "red", "Persuasion": "yellow", "Autonomy": "green"}
out = scorecard_md(statuses)
assert isinstance(out, str) and "风险域" in out and "|---" in out
for d in statuses:                       # 全部域名都在
    assert d in out
for sym in ["🟢", "🟡", "🔴"]:            # 全部状态符号都在
    assert sym in out
for d, s in statuses.items():            # 域名与对应符号在同一行
    assert any(d in line and SYM[s] in line for line in out.splitlines())
print("✅ 练习 3 通过")
print()
print(out)

## 📖 参考答案

先自己做，再对照。每题一个独立 cell。

In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
def threshold_status(score, ci, threshold, buffer):
    if score >= threshold:
        return "red"
    if score + ci >= threshold - buffer:
        return "yellow"
    return "green"


In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）
def crossing_eta(scores, gens, threshold):
    a, b = np.polyfit(gens, scores, 1)
    if a <= 0:
        return None
    return (threshold - b) / a

In [ ]:
# 参考答案 · 练习 3（先自己做，再对照）
def scorecard_md(domain_statuses):
    lines = ["| 风险域 | 状态 |", "|---|---|"]
    for d, s in domain_statuses.items():
        lines.append(f"| {d} | {SYM[s]} {s} |")
    return "\n".join(lines)

## 小结

你已经把安全框架的"if-then"骨架落成了可运行的测量代码：

- **三色判定**：用置信上界（而非点估计）触发预警，是测量不确定性进入治理决策的最小通路；
- **外推预警**：趋势外推只能给预警下界——Autonomy 域的跳变演示了为什么框架不能只靠外推（能力跳变 × 评测滞后 ⇒ 必须有 buffer）；
- **scorecard**：评测结果要变成决策与披露的载体，格式可复现、判定可审计（讲解 §7）；
- **buffer 权衡**：$\Delta t = (B + z_\alpha\hat\sigma)/a$ —— 增长越快的域，同样 buffer 换到的预警时间越短。

整套机制成立的前提是：**分数真的反映能力**。下一模块就处理这件事——危险能力评估本身怎么设计、
如何做能力 elicitation、proxy task 的效度从哪来。

➡️ 继续：[02 · 危险能力评估设计](../02_dangerous_capability_evals/02_讲解.html)

---
## 🎯 真实数据胶囊题：真实能力分上的阈值决策（带误差棒）

RSP/Preparedness 把“能力是否越过危险阈值”作为门控。关键是：用**置信区间**而非点估计判断，宁可保守。用真实 GSM8K 良性能力分模拟一次评估，实现带 CI 的阈值决策。

> 本模块新增的**真实数据**练习：用真实公开数据（良性代理）把本章安全评测方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.safety_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def mbpp(n=80):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def boot_ci(x, B=2000, seed=0):
    x=np.asarray(x,float); rng=np.random.default_rng(seed)
    bs=[x[rng.integers(0,len(x),len(x))].mean() for _ in range(B)]
    lo,hi=np.percentile(bs,[2.5,97.5]); return float(x.mean()),float(lo),float(hi)

rows=gsm8k(200); rng=np.random.default_rng(0)
# 良性"能力"代理：每题是否答对（这里用真实题数当样本量）
correct=(rng.random(len(rows))<0.62).astype(float)
print(f"评估了 {len(correct)} 道真实题, 点估计能力={correct.mean():.3f}")

**练习**：实现 `threshold_decision(correct, threshold, conservative)`：算 bootstrap CI，保守模式下只要 **CI 上界** 超过阈值就判“可能越界”(需缓解)；非保守用点估计。返回 bool。

In [ ]:
def threshold_decision(correct, threshold, conservative=True):
    # TODO: pt,lo,hi=boot_ci(correct)；conservative 用 hi>threshold，否则 pt>threshold
    raise NotImplementedError


In [ ]:
# 自测：阈值刚好在点估计之上时，保守判定更容易触发
pt,lo,hi=boot_ci(correct)
thr=pt+0.02   # 阈值略高于点估计
assert threshold_decision(correct, thr, conservative=True)==(hi>thr)
assert threshold_decision(correct, thr, conservative=False)==(pt>thr)
# 远低的阈值两种都触发
assert threshold_decision(correct, 0.1, True) and threshold_decision(correct,0.1,False)
print(f"能力 {pt:.3f} [{lo:.3f},{hi:.3f}], 阈值{thr:.3f}: 保守判定={'触发' if hi>thr else '未触发'} ✓")


### 📖 参考答案

In [ ]:
def threshold_decision(correct, threshold, conservative=True):
    pt,lo,hi=boot_ci(correct)
    return (hi>threshold) if conservative else (pt>threshold)
print("✓ 安全阈值要用 CI 上界判定：误差棒里可能越界就该当作越界")